In [1]:
import sys

sys.path.append("./lsr_package/")

from lsr.transformer import LSR
from pyterrier_pisa import PisaIndex, PisaToksIndexer
import pandas as pd
from tqdm import tqdm
import pyterrier as pt
if not pt.started():
    pt.init()
from ir_measures import nDCG
import os
import uuid
import pickle
import numpy as np
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
import torch

/opt/miniconda3/envs/splade_class/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_9790/3020091603.py:10: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_9790/3020091603.py:11: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

# Util functions
def ohsumed_iterator():
    for row in tqdm(docs.iterrows(), total = len(docs), desc = "Indexing"):
        row = row[1]
        yield {"docno" : row.docno, "text" : row.text}

def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [3]:
model_folder = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model/"
index_folder = "/nfs/primary/SPLADE/splade_class/indices/sens_reg_exp/splade_ohsumed_multiple_negative"
lsr = LSR(model_folder)

In [4]:
encoded_queries = lsr.encode_queries(queries["query"].tolist(), out_fmt = "np")

In [33]:
k = 100  # Set k locally
file = open("/nfs/primary/SPLADE/splade_class/svm_coefficients.pkl", "rb")
svm_vec = pickle.load(file)
file.close()

# Get the indices of the top-k values in the SVM vector
top_k_indices = np.argsort(svm_vec)[-k:]
encoded_queries[:,top_k_indices].mean(axis = 1)

array([0.00000000e+00, 1.60338115e-02, 1.14910947e-02, 1.18209841e-02,
       9.45328968e-04, 4.19684406e-03, 3.54921073e-03, 1.71417801e-03,
       4.68984842e-02, 0.00000000e+00, 6.36895141e-03, 0.00000000e+00,
       7.29762856e-03, 0.00000000e+00, 8.87497514e-03, 6.76630717e-03,
       5.25144814e-03, 1.73056824e-03, 8.59664287e-03, 1.61932269e-03,
       2.27519800e-03, 0.00000000e+00, 9.10793024e-04, 0.00000000e+00,
       4.22308408e-03, 1.28694100e-03, 2.06768233e-02, 2.07757782e-02,
       1.79779588e-03, 2.74684583e-03, 1.41284335e-02, 1.57452002e-02,
       1.73226614e-02, 5.20725735e-03, 0.00000000e+00, 9.07091703e-03,
       1.10345911e-02, 8.47629458e-03, 7.68296001e-03, 1.80233791e-02,
       1.18240691e-03, 3.03046196e-03, 6.04054285e-03, 1.09310150e-02,
       0.00000000e+00, 0.00000000e+00, 2.24554520e-02, 2.07624808e-02,
       5.95517829e-03, 0.00000000e+00, 9.82740312e-06, 1.75154041e-02,
       3.07951923e-02, 1.54168939e-03, 4.77687083e-03, 9.89629328e-03,
      

In [32]:
random_k_indices = np.random.randint(30522, size=k)
encoded_queries[:,random_k_indices].mean(axis = 1)

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00176854, 0.        , 0.        , 0.        , 0.02037242,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.     

In [37]:
file = open("/nfs/primary/SPLADE/splade_class/svm_coefficients.pkl", "rb")
svm_vec_v1 = pickle.load(file)

file = open("/nfs/primary/SPLADE/splade_class/svm_coefficients_v3.pkl", "rb")
svm_vec_v3 = pickle.load(file)

In [38]:
svm_vec_v1.max()

0.07174570571807377

In [39]:
svm_vec_v3.max()

0.07174783094989629